# Privacy budgets and accounting

The privacy budget represents how much privacy loss is allowed for the given statistical estimation problem.

Many private algorithms have multiple components, each with their own privacy cost. We need to account for these costs together to obtain the privacy guarantee of the full algorithm. An algorithm’s internal accounting may also use a different privacy definition from the allotted budget. For example, an algorithm offered an approximate-DP budget may account internally in zCDP or GDP.


To help handle this accounting, the library offers a number of privacy classes and some accounting helpers. Some of these helpers use Google’s [`dp_accounting`](https://github.com/google/differential-privacy/tree/main/python/dp_accounting) library.
This notebook goes through how to handle privacy budgets with this library.

The library implements the following budget classes, all extending the abstract class [DPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.DPBudget/):
- [`PureDPBudget`](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.PureDPBudget/): pure differential privacy.
- [`ApproxDPBudget`](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.ApproxDPBudget/): approximate differential privacy.
- [`ZCDPBudget`](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.ZCDPBudget/): zero-concentrated differential privacy.
- [`GDPBudget`](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.GDPBudget/): Gaussian differential privacy.

[`NO_PRIVACY`](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.NoPrivacy/) requests an unconstrained release.

You can implement your own budget class by extending [DPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.DPBudget/) and implementing the `split` method and `is_unconstrained` property.



In [ ]:
from dpstats.budgets import (
    NO_PRIVACY,
    ApproxDPBudget,
    GDPBudget,
    PureDPBudget,
    ZCDPBudget,
    accounting,
)
from dpstats.exceptions import InvalidRegime

pure_budget = PureDPBudget(eps=1.0)
approx_budget = ApproxDPBudget(eps=1.0, delta=1e-6)
zcdp_budget = ZCDPBudget(rho=0.1)
declared_gdp_budget = GDPBudget(mu=0.5)

print("pure DP:", pure_budget)
print("approximate DP:", approx_budget)
print("zCDP:", zcdp_budget)
print("GDP:", declared_gdp_budget)
print("unconstrained:", NO_PRIVACY.is_unconstrained)


## Accounting in the Gaussian mechanism


To illustrate accounting, we return to the [GaussianMechanism](https://saminulh.github.io/dpstats/mean_estimation/algorithms/GaussianMechanism/) example introduced in the [Algorithms notebook](https://saminulh.github.io/dpstats/library/tutorials/algorithms/).

We recall that the Gaussian mechanism simply called the [accounting.calibrate_gaussian_scale](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.accounting.calibrate_gaussian_scale/) function to calibrate its noise scale.
Looking inside that function, we see the following implementation:
```python
import math

from dpstats.budgets import DPBudget
from dpstats.budgets.accounting import spend_as_gdp


def calibrate_gaussian_scale(budget: DPBudget, sensitivity: float) -> float:
    if not math.isfinite(sensitivity) or sensitivity <= 0:
        raise ValueError("sensitivity must be finite and positive.")
    return sensitivity / spend_as_gdp(budget).mu
```

This implementation calibrates its noise natively using Gaussian differential privacy (GDP). For $\mu$-GDP, the Gaussian mechanism adds noise with scale $\Delta_2/\mu$, where $\Delta_2$ is the query’s $\ell_2$-sensitivity.

However, the `budget` argument may be any [DPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.DPBudget/) type, not just [GDPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.GDPBudget/). Rather than rejecting other supported budget types or hand-coding conversions, the function uses [spend_as_gdp](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.accounting.spend_as_gdp/) to obtain a `GDPBudget` whose guarantee is sufficient for the supplied budget.

Below we show an example of converting an approx-DP budget to a Gaussian-DP budget.

In [ ]:
gdp_budget = accounting.spend_as_gdp(approx_budget)
print("Offered budget:", approx_budget)
print("GDP budget available to the mechanism:", gdp_budget)


However, no finite [GDPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.GDPBudget/) provides a guarantee sufficient for a finite pure-DP budget. Attempting to spend a finite pure-DP budget as GDP therefore raises an [InvalidRegime](https://saminulh.github.io/dpstats/library/reference/exceptions/dpstats.exceptions.InvalidRegime/) error, meaning the supplied budget is invalid for the given algorithm.

In [ ]:
try:
    accounting.spend_as_gdp(pure_budget)
except InvalidRegime as error:
    print(f"{type(error).__name__}: {error}")


We can now run the mechanism on synthetic data. [`BoxDomain`](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.BoxDomain/) supplies the public bound (see the [Domain notebook](https://saminulh.github.io/dpstats/library/tutorials/domains/) for how domains interact with budgets). The fixed seed is for reproducibility, not a private deployment.

In [ ]:
import numpy as np

from dpstats.domains import BoxDomain
from dpstats.tasks.mean_estimation import (
    MeanEstimationInput,
    MeanEstimationMetadata,
)
from dpstats.tasks.mean_estimation.algorithms import GaussianMechanism

X = np.array([[0.1], [0.3], [0.6], [0.8]])
metadata = MeanEstimationMetadata(
    n_samples=len(X), domain=BoxDomain(bounds=np.array([[0.0, 1.0]]))
)
input = MeanEstimationInput(X=X)
metadata.validate_input(input)
mechanism = GaussianMechanism()
output = mechanism.run(metadata, input, approx_budget, seed=2026)
print("Output on an approx-dp budget:", output.mean_est)


## Accounting helpers

The library provides accounting helpers for common private primitives. See the [accounting](https://saminulh.github.io/dpstats/library/reference/budgets/#dpstats.budgets.accounting) API for the available functions.

For example, [accounting.calibrate_gaussian_scale](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.accounting.calibrate_gaussian_scale/) computes the Gaussian noise scale directly from the budget and sensitivity, without requiring us to convert the budget ourselves:

In [ ]:
_, radius = metadata.domain.l2_enclosing_ball()
sensitivity = radius / metadata.n_samples
sigma = accounting.calibrate_gaussian_scale(approx_budget, sensitivity)
print("Noise scale from accounting:", sigma)
print("Noise scale from GDP conversion:", sensitivity / gdp_budget.mu)


## Splitting a budget

When an algorithm has multiple private steps, we account for its overall privacy by composing the guarantees of those steps. We can allocate the total budget among them using [DPBudget.split](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.DPBudget/#dpstats.budgets.DPBudget.split).

Call `split` on the total budget, passing the relative weights for the stages.

Here we split `gdp_budget` into two equal shares. Running the mechanism with these shares ensures that releasing both `first` and `second` stays within the original `gdp_budget`.

In [ ]:
first_budget, second_budget = gdp_budget.split([0.5, 0.5])
first = mechanism.run(metadata, input, first_budget, seed=2026)
second = mechanism.run(metadata, input, second_budget, seed=2027)

print("original budget:", gdp_budget)
print("split budgets:", first_budget, second_budget)
print("first output:", first.mean_est)
print("second output:", second.mean_est)


## Next steps

Next: [Metrics and baselines](https://saminulh.github.io/dpstats/library/tutorials/metrics_and_baselines/).
